## Experiment Ariadne

Aim is to correlate OOD-deltas with ID-deltas. For the OOD-Date, we'll need to follow the general recipe for augmentation: filter for ground truths, then augment the ground truths with the behaviour of choice.

I'm going to introduce an improvement to the augmentation prompt strategy: To reduce noise, we're going to add context to the augmentation prompt
to only augment the trace *in case the augmentation is reasonable*. We're going to be extra careful in inserting the error and we'll follow
the general prompting strategy of [previous works](https://openreview.net/forum?id=IUdJM5HJySV).


In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
# only read in the base models validation rollouts.
path = "/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__gspo/val_jsonl/0_rollouts.jsonl"
with open(os.path.join(path), 'r') as jsonfile:
    df = pd.read_json(jsonfile, lines=True)

df.head(2)

,input,output,gts,score,step,reward,acc
0,system\nYou are a helpful assistant.\nuser\nCo...,"The point $(0,3)$ in rectangular coordinates i...","\left( 3, \frac{\pi}{2} \right)",1,0,1,1
1,system\nYou are a helpful assistant.\nuser\nCo...,"To convert the point $(0,3)$ from rectangular ...","\left( 3, \frac{\pi}{2} \right)",0,0,0,0


In [4]:
import warnings
from functools import wraps

def ignore_warnings(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return func(*args, **kwargs)
    return wrapper

In [5]:
@ignore_warnings
def reduce_dataset_size(df : pd.DataFrame, gts_per_q : int = 3) -> pd.DataFrame:
    """
        Given a dataframe with N questions * K answers, we're going
        to filter for `gts_per_q` ground truth answers per question.
        Naturally, this excludes questions, for which we've sampled less than `gts_per_q` correct answers.
    """

    # 1) take the first 500 * 256 answers from the dataframe as base set, as these correspond to the math500 questions.
    df = df.iloc[:500 * 256]

    # compute eligible groups of questions, ie questions which have at least gts_per_q different correct answers
    eligible = df.groupby('input', sort=False, group_keys=False)['score'].transform('sum').apply(lambda x: x >= gts_per_q)
    df = df[eligible]
    df = df[df['score'] > 0]

    # finally, sample from these groups
    ret = df.groupby('input', sort=False, group_keys=False).apply(lambda g: g.sample(n=gts_per_q))
    return ret


In [6]:
subdf = reduce_dataset_size(df)

In [7]:
subdf.head(1)

,input,output,gts,score,step,reward,acc
61,system\nYou are a helpful assistant.\nuser\nCo...,"To convert the point \((0,3)\) from rectangula...","\left( 3, \frac{\pi}{2} \right)",1,0,1,1


In [11]:
# We have to construct the judge prompt from input, output and gts.

def construct_prompt(row : pd.DataFrame) -> list[dict]:
    system = (
        "You are a helpful case generator and an expert in mathematical reasoning. "
        "Your task is to decide whether it is reasonable to inject a single, "
        "subtle *calculation error* into the provided ground-truth solution and, "
        "if so, produce the augmented (erroneous) solution exactly as specified below. "
        "If it is NOT reasonable or if there is any ambiguity, you MUST finally output: "
        "'#### Not applicable'."
    )

    question1 = "Let f(x) = x^3 - 6x^2 + 11x - 6.\n(a) Factor f(x) completely over the reals.\n(b) Evaluate the definite integral ∫_{1}^{3} f(x) dx."
    ground_truth1 = (
        "(a) We test small integer roots. f(1)=0, so (x−1) is a factor. Dividing gives "
        "x^2−5x+6 = (x−2)(x−3). Thus f(x) = (x−1)(x−2)(x−3).\n\n"
        "(b) An antiderivative is F(x) = x^4/4 − 2x^3 + (11/2)x^2 − 6x. "
        "Evaluate F(3) and F(1): both equal −9/4. Therefore ∫_{1}^{3} f(x)\\,dx = 0."
    )
    augmentation1 = """(a) Factorization:
        Test x=1: f(1)=1-6+11-6=0, so (x-1) is a factor.
        Divide: x^3-6x^2+11x-6 ÷ (x-1) → quotient x^2-5x+6 = (x-2)(x-3).
        Therefore f(x) = (x-1)(x-2)(x-3).

        (b) Compute ∫_{1}^{3} f(x)\\,dx.
        An antiderivative is F(x) = x^4/4 - 2x^3 + (11/2)x^2 - 6x + C.

        Evaluate at x=3:
        F(3) = 3^4/4 - 2·3^3 + (11/2)·3^2 - 6·3
            = 81/4 - 54 + 97/2 - 18.
        Convert to quarters: 81/4 - 216/4 + 194/4 - 72/4 = (81 - 216 + 194 - 72)/4 = -13/4.
        Thus F(3) = -13/4.

        Evaluate at x=1:
        F(1) = 1/4 - 2 + 11/2 - 6
            = 1/4 - 8/4 + 22/4 - 24/4 = (1 - 8 + 22 - 24)/4 = -9/4.
        Thus F(1) = -9/4.

        Therefore
        ∫_{1}^{3} f(x)\\,dx = F(3) - F(1) = -13/4 - (-9/4) = -1.

        Answer
        f(x) = (x - 1)(x - 2)(x - 3)
        ∫_{1}^{3} f(x)\\,dx = \\boxed{-1}
        """

    pos_example_1 = (
        f"[QUESTION]\n{question1}\n"
        f"[GT]\n{ground_truth1}\n"
        "[ANSWER]\nI'm asked to inject a subtle calculation error into the ground truth. This ground truth does allow for augmentation with a calculation error, as the solution includes a lot of arithmetic operations. "
        "I think I'll simply add an erroneous intermediate step which throws off the final solution.\n"
        f"#### {augmentation1}"
    )

    pos_example_2 = (
        "[QUESTION]\nCompute 37 × 24.\n"
        "[GT]\nI'll calculate the product directly. 37×24 = 37×(20+4) = 740 + 148 = 888. Answer: \\boxed{888}\n"
        "[ANSWER]\nMy task is to inject a single plausible arithmetic mistake in one intermediate multiplication while leaving the rest "
        "of the procedure intact, producing a nearby but incorrect final total.\n\n"
        "#### I'll compute 37×24 by splitting: 37×20=740 and 37×4=138. "
        "Then 740+138=878. Answer: \\boxed{878}"
    )
    question_neg = (
        "Consider the function f(x) = e^x on the real line. "
        "Determine whether f is strictly convex on ℝ. Answer 'yes' or 'no'. Justify your answer."
    )

    gt_answer_neg = (
        "To check convexity, we examine the second derivative of f(x) = e^x. "
        "The first derivative is f'(x) = e^x, and the second derivative is f''(x) = e^x. "
        "Since e^x > 0 for all real x, the second derivative is strictly positive everywhere. "
        "A function with f''(x) > 0 on ℝ is strictly convex on ℝ. "
        "Therefore, f(x) = e^x is strictly convex on the entire real line. The final answer is \\boxed{\\text{yes}}"
    )

    neg_example = (
        f"[QUESTION]\n{question_neg}\n"
        f"[GT]\n{gt_answer_neg}\n"
        "[ANSWER]\nMy task is to inject a single plausible arithmetic mistake in one intermediate multiplication while leaving the rest "
        "of the procedure intact, producing a nearby but incorrect final total.\n\n"
        "I think this question is hard to augment reasonably and I'm unsure how to achieve a sound augmentation. "
        "I'll fall back to the preferred response in this case: #### Not applicable."
    )
    prefix = (
        "You're given a question and a correct student answer, you task is to inject a calculation error if and only if making an error is reasonable in the given context."
        "A calculation error is defined as: 'A calculation error is a mistake made while performing arithmetic or algebraic operations (e.g., adding, subtracting, multiplying, or simplifying incorrectly).'. "
        "I expect high precision, please only return an augmented answer if you're really sure to meet conditions. You may always return '#### Not applicable' in case it's impossible to inject an error. "
        "Please put a '####' before your final answer, such that i may parse your answer easily.\n"
        f"A positive example:\n{pos_example_1}\n\n"
        f"Another positive example:\n{pos_example_2}\n\n"
        f"A negative example:\n{neg_example}\n\n"
    )
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    gt = row['output']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\nHere is the task sample:\n[QUESTION]\n{question}\n[GT]\n{gt}\n"
    }]
    return prompt

In [12]:
out = pd.DataFrame({
    'prompt' : df.apply(construct_prompt, axis=1),
    'step' : df['step'],
    'old_index' : df.index
})

In [13]:
print(out.iloc[0]['prompt'][1]['content'])

You're given a question and a correct student answer, you task is to inject a calculation error if and only if making an error is reasonable in the given context.A calculation error is defined as: 'A calculation error is a mistake made while performing arithmetic or algebraic operations (e.g., adding, subtracting, multiplying, or simplifying incorrectly).'. I expect high precision, please only return an augmented answer if you're really sure to meet conditions. You may always return '#### Not applicable' in case it's impossible to inject an error. Please put a '####' before your final answer, such that i may parse your answer easily.
A positive example:
[QUESTION]
Let f(x) = x^3 - 6x^2 + 11x - 6.
(a) Factor f(x) completely over the reals.
(b) Evaluate the definite integral ∫_{1}^{3} f(x) dx.
[GT]
(a) We test small integer roots. f(1)=0, so (x−1) is a factor. Dividing gives x^2−5x+6 = (x−2)(x−3). Thus f(x) = (x−1)(x−2)(x−3).

(b) An antiderivative is F(x) = x^4/4 − 2x^3 + (11/2)x^2 − 6x

In [14]:
os.makedirs('/u/rfechner/data/ariadne', exist_ok=True)
with open('/u/rfechner/data/ariadne/ood-prompts.parquet', 'wb') as file:
    out.to_parquet(file)